## Combining Sentiment and Market Data

I have now undertaken the sentiment analysis of news articles and have explored the returns over the same time range. Now I want to combine the data in the format of input and output and split this data into a training (training + dev-test?) as well as the test set (2021).


* Combine relevant market data
     * (Regression or Classification) Probably too noisy to look at % gains. Maybe boolean for gain over next 1,3,5 day period?
* Look at what to include in terms of sentiment
    - Try with negative and compound
    - Positive and compound
    - All the tags

In [29]:
import pandas as pd
import numpy as np

In [11]:
daily_sentiments = pd.read_csv('../sentiment_analysis/daily_sentiments.csv')

In [12]:
daily_sentiments

,date,title_neg,title_neu,title_pos,title_compound,desc_neg,desc_neu,desc_pos,desc_compound
0,2021-04-16,0.092604,0.793604,0.113792,0.025758,0.069083,0.842917,0.087979,0.067115
1,2021-04-15,0.062340,0.840470,0.097190,0.053673,0.054310,0.856250,0.089450,0.156695
2,2021-04-14,0.098358,0.817979,0.083674,-0.026956,0.064200,0.859800,0.076021,0.104177
3,2021-04-13,0.052677,0.869277,0.078015,0.051142,0.056985,0.867354,0.075646,0.106286
4,2021-04-12,0.049151,0.837890,0.112959,0.094377,0.059178,0.864027,0.076808,0.040249
...,...,...,...,...,...,...,...,...,...
1118,2018-03-24,0.101455,0.806364,0.092182,0.005636,0.070364,0.861818,0.067818,-0.077764
1119,2018-03-23,0.101315,0.801315,0.097370,-0.011381,0.063222,0.843093,0.093704,0.155633
1120,2018-03-22,0.117712,0.794685,0.087616,-0.034803,0.063425,0.859425,0.077082,0.056668
1121,2018-03-21,0.076710,0.839177,0.084113,0.014061,0.047355,0.875855,0.076774,0.115885


In order not to work with too many columns I will be combining the sentiments from the title and the description

In [24]:
daily_sentiments['combined compound'] = (daily_sentiments['title_compound'] + daily_sentiments['desc_compound']) / 2
daily_sentiments

,date,title_neg,title_neu,title_pos,title_compound,desc_neg,desc_neu,desc_pos,desc_compound,combined compound
0,2021-04-16,0.092604,0.793604,0.113792,0.025758,0.069083,0.842917,0.087979,0.067115,0.046436
1,2021-04-15,0.062340,0.840470,0.097190,0.053673,0.054310,0.856250,0.089450,0.156695,0.105184
2,2021-04-14,0.098358,0.817979,0.083674,-0.026956,0.064200,0.859800,0.076021,0.104177,0.038611
3,2021-04-13,0.052677,0.869277,0.078015,0.051142,0.056985,0.867354,0.075646,0.106286,0.078714
4,2021-04-12,0.049151,0.837890,0.112959,0.094377,0.059178,0.864027,0.076808,0.040249,0.067313
...,...,...,...,...,...,...,...,...,...,...
1118,2018-03-24,0.101455,0.806364,0.092182,0.005636,0.070364,0.861818,0.067818,-0.077764,-0.036064
1119,2018-03-23,0.101315,0.801315,0.097370,-0.011381,0.063222,0.843093,0.093704,0.155633,0.072126
1120,2018-03-22,0.117712,0.794685,0.087616,-0.034803,0.063425,0.859425,0.077082,0.056668,0.010933
1121,2018-03-21,0.076710,0.839177,0.084113,0.014061,0.047355,0.875855,0.076774,0.115885,0.064973


In [25]:
daily_sentiments['combined negative'] = (daily_sentiments['title_neg'] + daily_sentiments['desc_neg']) / 2
daily_sentiments['combined positive'] = (daily_sentiments['title_pos'] + daily_sentiments['desc_pos']) / 2

daily_sentiments = daily_sentiments[['date', 'combined compound', 'combined positive', 'combined negative']]
daily_sentiments

,date,combined compound,combined positive,combined negative
0,2021-04-16,0.046436,0.100885,0.080844
1,2021-04-15,0.105184,0.093320,0.058325
2,2021-04-14,0.038611,0.079847,0.081279
3,2021-04-13,0.078714,0.076831,0.054831
4,2021-04-12,0.067313,0.094884,0.054164
...,...,...,...,...
1118,2018-03-24,-0.036064,0.080000,0.085909
1119,2018-03-23,0.072126,0.095537,0.082269
1120,2018-03-22,0.010933,0.082349,0.090568
1121,2018-03-21,0.064973,0.080444,0.062032


In order to get a performance measure, I will be looking at the market returns 1 day ahead, 3 days ahead, 5 days ahead, and 10 days ahead.

In [95]:
combined_market_data = pd.read_csv('../data/calculations/combined_market_data.csv')
combined_market_data = combined_market_data[['Date','Formatted Change','Volume', 'Annualised']]
combined_market_data.columns = ['Date', 'Close-Close Change', 'Volume', 'Annualised 30D Volatility']
combined_market_data

,Date,Close-Close Change,Volume,Annualised 30D Volatility
0,2017-01-04,1.005722,3764890000,NaN
1,2017-01-05,0.999229,3761820000,NaN
2,2017-01-06,1.003517,3339890000,NaN
3,2017-01-09,0.996451,3217610000,NaN
4,2017-01-10,1.000000,3638790000,NaN
...,...,...,...,...
1073,2021-04-12,0.999804,3578500000,0.152654
1074,2021-04-13,1.003295,3728440000,0.139007
1075,2021-04-14,0.995912,3976540000,0.136951
1076,2021-04-15,1.011094,4027680000,0.131398


The vol was done on the 30D of the market, the 1,3,5, and 10 day performance will also be using the market open days compared to the whole year.

In [96]:
# Next day
combined_market_data['Future 1D'] = np.nan
for i in range(0,1077):
    combined_market_data['Future 1D'].iloc[i] = combined_market_data['Close-Close Change'].iloc[i+1]
combined_market_data

/Users/oenmalm/anaconda3/lib/python3.8/site-packages/pandas/core/indexing.py:671: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_with_indexer(indexer, value)


,Date,Close-Close Change,Volume,Annualised 30D Volatility,Future 1D
0,2017-01-04,1.005722,3764890000,NaN,0.999229
1,2017-01-05,0.999229,3761820000,NaN,1.003517
2,2017-01-06,1.003517,3339890000,NaN,0.996451
3,2017-01-09,0.996451,3217610000,NaN,1.000000
4,2017-01-10,1.000000,3638790000,NaN,1.002830
...,...,...,...,...,...
1073,2021-04-12,0.999804,3578500000,0.152654,1.003295
1074,2021-04-13,1.003295,3728440000,0.139007,0.995912
1075,2021-04-14,0.995912,3976540000,0.136951,1.011094
1076,2021-04-15,1.011094,4027680000,0.131398,1.003609


In [97]:
# Simple impementation. For the 3 data points we need we could lower the amount of computation required by taking the already 
# calculated and compounding the next x along with it. For code comprehension and ease there is redundancy.
def future_forecast(num_days):
    column_name = 'Future {}D'.format(num_days)
    combined_market_data[column_name] = np.nan
    for i in range(0, len(combined_market_data) - (num_days)):
        # Takes the next num_days and takes the value of the cumulative product of the last day
        result = combined_market_data['Close-Close Change'].iloc[i + 1: i + 1 + num_days].cumprod()[i + num_days]
        combined_market_data[column_name].iloc[i] = result
        
        
future_forecast(3)
future_forecast(5)
future_forecast(10)
combined_market_data

/Users/oenmalm/anaconda3/lib/python3.8/site-packages/pandas/core/indexing.py:671: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_with_indexer(indexer, value)


,Date,Close-Close Change,Volume,Annualised 30D Volatility,Future 1D,Future 3D,Future 5D,Future 10D
0,2017-01-04,1.005722,3764890000,NaN,0.999229,0.999185,1.002013,0.996891
1,2017-01-05,0.999229,3761820000,NaN,1.003517,0.999956,1.000635,1.001018
2,2017-01-06,1.003517,3339890000,NaN,0.996451,0.999271,0.998972,0.994826
3,2017-01-09,0.996451,3217610000,NaN,1.000000,1.000679,0.999555,1.004923
4,2017-01-10,1.000000,3638790000,NaN,1.002830,1.002530,1.001318,1.012989
...,...,...,...,...,...,...,...,...
1073,2021-04-12,0.999804,3578500000,0.152654,1.003295,1.010279,NaN,NaN
1074,2021-04-13,1.003295,3728440000,0.139007,0.995912,1.010595,NaN,NaN
1075,2021-04-14,0.995912,3976540000,0.136951,1.011094,NaN,NaN,NaN
1076,2021-04-15,1.011094,4027680000,0.131398,1.003609,NaN,NaN,NaN


In [99]:
model_data = combined_market_data.copy()

In [ ]:
model_data.set_index('Date', inplace=True)
